# 🔬 **Silicon Photonics Measurement Analysis**

**Project:** HY202103 Silicon Photonics Measurement Data Analysis  
**Targets:** GPDO (Germanium Photodetector) / MZM (Mach-Zehnder Modulator — LMZC / LMZO)

<table style="width:55%; border-collapse:collapse; font-size:15px; margin-top:10px;">
  <thead>
    <tr style="background-color:#3a3a3a; color:white;">
      <th style="padding:10px 24px; text-align:center;">Wafer</th>
      <th style="padding:10px 24px; text-align:center;">GPDO</th>
      <th style="padding:10px 24px; text-align:center;">MZM</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#555555; color:white;">
      <td style="padding:9px 24px; text-align:center; font-weight:bold;">D07</td>
      <td style="padding:9px 24px; text-align:center; color:#aaa;">✗</td>
      <td style="padding:9px 24px; text-align:center;">✅ LMZC</td>
    </tr>
    <tr style="background-color:#666666; color:white;">
      <td style="padding:9px 24px; text-align:center; font-weight:bold;">D08</td>
      <td style="padding:9px 24px; text-align:center;">✅</td>
      <td style="padding:9px 24px; text-align:center;">✅ LMZC · LMZO</td>
    </tr>
    <tr style="background-color:#555555; color:white;">
      <td style="padding:9px 24px; text-align:center; font-weight:bold;">D23</td>
      <td style="padding:9px 24px; text-align:center;">✅</td>
      <td style="padding:9px 24px; text-align:center;">✅ LMZO</td>
    </tr>
    <tr style="background-color:#666666; color:white;">
      <td style="padding:9px 24px; text-align:center; font-weight:bold;">D24</td>
      <td style="padding:9px 24px; text-align:center;">✅</td>
      <td style="padding:9px 24px; text-align:center;">✅ LMZO</td>
    </tr>
  </tbody>
</table>

> ⚠ Some timestamps contain only alignment/calibration data (no GPDO/MZM files).  
> If a timestamp shows **"No files found"**, select a different timestamp.

---
### ✅ How to use

- **Step 1** — Environment setup
- **Step 2** — Select analysis conditions (device, wafer, timestamp, die)
- **Step 3** — Data Loading & XML Parsing
- **Step 4** — Analysis results table
- **Step 5** — Visualization
- **Step 6** — Save results


---
## Step 1 — Environment Setup

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Step 1 : Environment Setup
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, sys, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.dirname(os.path.abspath('__file__'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, HTML as _HTML

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

from config import DATA_DIR, RES_DIR, PROJECT_NAME
from src.gpdo.parser    import GPDOParser
from src.gpdo.fitting   import FittingEngine
from src.mzm.parser     import MZMParser
from src.mzm.fitting    import fit_mzi, parse_array, process_spectrum, process_iv

display(_HTML(f'''
<div style="margin-top:10px; padding:12px 18px; border-radius:8px;
            background:#1a2233; border:1.5px solid #2a4a7a;
            font-size:13px; line-height:2.0;">
  <span style="font-size:14px; font-weight:bold; color:#aaccff;">
    ✅ &nbsp;Environment ready
  </span><br>
  <span style="color:#777;">Project</span>&nbsp;&nbsp;&nbsp;&nbsp;
    <b style="color:white;">{PROJECT_NAME}</b><br>
  <span style="color:#777;">Data path</span>&nbsp;&nbsp;
    <span style="color:#aaccff;">{DATA_DIR}</span><br>
  <span style="color:#777;">Save path</span>&nbsp;&nbsp;
    <span style="color:#aaccff;">{RES_DIR}</span>
</div>'''))

---
## Step 2 — Analysis Condition Selection

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Step 2 : Analysis Condition Selection (Widgets)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Device → supported wafer mapping ─────────────────
DEVICE_WAFER_MAP = {
    'GPDO': ['D08', 'D23', 'D24'],
    'MZM' : ['D07', 'D08', 'D23', 'D24'],
}

# ── Selection state ───────────────────────────────────
selection = dict(device=None, wafer=None, timestamp=None,
                 die=None, xml_path=None)

# ── Widget layout ─────────────────────────────────────
dd_layout    = widgets.Layout(width='340px')
badge_layout = widgets.Layout(width='220px', margin='0 0 0 10px')
style        = {'description_width': '140px'}

# ── Badge HTML helpers ────────────────────────────────
def _badge_pending(label='not selected'):
    return (f'<span style="background:#3a3a3a; color:#777777; '
            f'padding:4px 14px; border-radius:20px; font-size:12px; '
            f'font-weight:bold;">● {label}</span>')

def _badge_done(label):
    return (f'<span style="background:#1a5c1a; color:#aeffae; '
            f'padding:4px 14px; border-radius:20px; font-size:12px; '
            f'font-weight:bold;">✔ {label}</span>')

def _badge_warn(label):
    return (f'<span style="background:#7a1010; color:#ffaaaa; '
            f'padding:4px 14px; border-radius:20px; font-size:12px; '
            f'font-weight:bold;">✘ {label}</span>')

# ── Dropdowns ─────────────────────────────────────────
w_device = widgets.Dropdown(
    options=['--- Select ---', 'GPDO', 'MZM'],
    description='① Device',
    layout=dd_layout, style=style,
)
w_wafer = widgets.Dropdown(
    options=['--- Select device first ---'],
    description='② Wafer',
    layout=dd_layout, style=style,
)
w_ts = widgets.Dropdown(
    options=['--- Select wafer first ---'],
    description='③ Timestamp',
    layout=dd_layout, style=style,
)
w_die = widgets.Dropdown(
    options=['--- Select timestamp first ---'],
    description='④ Die (col, row)',
    layout=dd_layout, style=style,
)

# ── Status badges ─────────────────────────────────────
b_device = widgets.HTML(value=_badge_pending(), layout=badge_layout)
b_wafer  = widgets.HTML(value=_badge_pending(), layout=badge_layout)
b_ts     = widgets.HTML(value=_badge_pending(), layout=badge_layout)
b_die    = widgets.HTML(value=_badge_pending(), layout=badge_layout)

# ── Summary panel ─────────────────────────────────────
out_summary = widgets.HTML(value='')

def _update_summary():
    d  = selection.get('device')    or '—'
    w  = selection.get('wafer')     or '—'
    t  = selection.get('timestamp') or '—'
    di = selection.get('die')       or '—'
    fn = os.path.basename(selection.get('xml_path') or '') or '—'
    all_done = all(selection.get(k) for k in ('device', 'wafer', 'timestamp', 'die'))
    if not all_done:
        out_summary.value = ''
        return
    out_summary.value = f'''
<div style="margin-top:14px; padding:12px 18px; border-radius:8px;
            background:#1e2a1e; border:1.5px solid #2e5e2e;
            font-size:13px; line-height:2.0;">
  <span style="font-size:14px; font-weight:bold; color:#aeffae;">
    ✅ &nbsp;Selection complete — ready to run Step 3
  </span><br>
  <span style="color:#777;">Device</span>&nbsp;&nbsp;
    <b style="color:white;">{d}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Wafer</span>&nbsp;&nbsp;
    <b style="color:white;">{w}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Timestamp</span>&nbsp;&nbsp;
    <b style="color:white;">{t}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Die</span>&nbsp;&nbsp;
    <b style="color:white;">{di}</b><br>
  <span style="color:#777;">File</span>&nbsp;&nbsp;&nbsp;&nbsp;
    <span style="color:#aaffaa;">{fn}</span>
</div>'''

# ── File scan helper ──────────────────────────────────
def scan_files(device, wafer, ts):
    ts_dir = os.path.join(DATA_DIR, wafer, ts)
    result = {}
    for fname in sorted(os.listdir(ts_dir)):
        fpath = os.path.join(ts_dir, fname)
        if not fname.lower().endswith('.xml'):
            continue
        if device == 'GPDO' and GPDOParser.is_gpdo_xml(fpath):
            import re
            m = re.search(r'\((-?\d+),(-?\d+)\)', fname)
            label = f'({m.group(1)}, {m.group(2)})' if m else fname
            result[label] = fpath
        elif device == 'MZM' and MZMParser.MZM_FILENAME_TAG in fname:
            import re
            m = re.search(r'\((-?\d+),(-?\d+)\)', fname)
            label = f'({m.group(1)}, {m.group(2)})' if m else fname
            result[label] = fpath
    return result

# ── Callbacks ─────────────────────────────────────────
def on_device_change(change):
    if change['new'] == '--- Select ---':
        b_device.value = _badge_pending(); return
    dev = change['new']
    selection['device'] = dev
    selection['wafer'] = selection['timestamp'] = selection['die'] = None
    b_device.value = _badge_done(dev)
    b_wafer.value  = _badge_pending()
    b_ts.value     = _badge_pending()
    b_die.value    = _badge_pending()
    w_wafer.options = ['--- Select ---'] + DEVICE_WAFER_MAP[dev]
    w_ts.options    = ['--- Select wafer first ---']
    w_die.options   = ['--- Select timestamp first ---']
    _update_summary()

def on_wafer_change(change):
    if 'Select' in change['new']: return
    wid = change['new']
    selection['wafer'] = wid
    selection['timestamp'] = selection['die'] = None
    b_wafer.value = _badge_done(wid)
    b_ts.value    = _badge_pending()
    b_die.value   = _badge_pending()
    wafer_dir = os.path.join(DATA_DIR, wid)
    tss = sorted([d for d in os.listdir(wafer_dir)
                  if os.path.isdir(os.path.join(wafer_dir, d))])
    w_ts.options  = ['--- Select ---'] + tss
    w_die.options = ['--- Select timestamp first ---']
    _update_summary()

def on_ts_change(change):
    if 'Select' in change['new']: return
    ts = change['new']
    selection['timestamp'] = ts
    selection['die'] = None
    die_map = scan_files(selection['device'], selection['wafer'], ts)
    selection['_die_map'] = die_map
    if die_map:
        b_ts.value    = _badge_done(ts)
        b_die.value   = _badge_pending(f'{len(die_map)} dies found')
        w_die.options = ['--- Select ---', 'All'] + list(die_map.keys())
    else:
        b_ts.value    = _badge_warn('no files')
        b_die.value   = _badge_pending()
        w_die.options = [f'❌ No {selection["device"]} files found']
    _update_summary()

def on_die_change(change):
    if 'Select' in change['new'] or '❌' in change['new']: return
    label = change['new']
    die_map = selection.get('_die_map', {})
    xml_path = die_map.get(label)
    selection['die']      = label
    selection['xml_path'] = xml_path
    b_die.value = _badge_done(label)
    _update_summary()

w_device.observe(on_device_change, names='value')
w_wafer.observe(on_wafer_change,   names='value')
w_ts.observe(on_ts_change,         names='value')
w_die.observe(on_die_change,       names='value')

# ── Display ───────────────────────────────────────────
row_layout = widgets.Layout(align_items='center', margin='3px 0')
display(widgets.VBox([
    widgets.HBox([w_device, b_device], layout=row_layout),
    widgets.HBox([w_wafer,  b_wafer],  layout=row_layout),
    widgets.HBox([w_ts,     b_ts],     layout=row_layout),
    widgets.HBox([w_die,    b_die],    layout=row_layout),
    out_summary,
]))

---
## Step 3 — Data Loading & XML Parsing

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Step 3 : Data Loading & XML Parsing
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from IPython.display import HTML as _HTML
try:
    from tqdm.notebook import tqdm as _tqdm
    _TQDM = True
except ImportError:
    _TQDM = False
    def _tqdm(x, **kw): return x

parsed_data_list = []  # ★ Reset on every run

DEVICE_TYPE = selection.get('device')
WAFER_ID    = selection.get('wafer')
IS_ALL      = (selection.get('die') == 'All')
ts          = selection['timestamp']
ts_dir      = os.path.join(DATA_DIR, WAFER_ID, ts)

if not DEVICE_TYPE or not WAFER_ID or not ts:
    raise ValueError('Please select Device → Wafer → Timestamp → Die in Step 2.')

# ── Determine XML files to parse ─────────────────────
if IS_ALL:
    if DEVICE_TYPE == 'GPDO':
        xml_paths = sorted([
            os.path.join(ts_dir, f) for f in os.listdir(ts_dir)
            if f.lower().endswith('.xml') and GPDOParser.is_gpdo_xml(os.path.join(ts_dir, f))
        ])
    else:
        xml_paths = sorted([
            os.path.join(ts_dir, f) for f in os.listdir(ts_dir)
            if f.lower().endswith('.xml') and MZMParser.MZM_FILENAME_TAG in f
        ])
    print(f'[All mode] {len(xml_paths)} files found — starting parse...\n')
else:
    xml_path = selection.get('xml_path')
    if not xml_path or not os.path.exists(xml_path):
        raise ValueError('Please select a Die in Step 2.')
    xml_paths = [xml_path]

# ── Parse ─────────────────────────────────────────────
ok_count = err_count = 0
_iter = _tqdm(xml_paths, desc=f'Parsing {DEVICE_TYPE}', unit='file',
              disable=(not IS_ALL or not _TQDM))

for _xml_path in _iter:
    _fname = os.path.basename(_xml_path)
    try:
        if DEVICE_TYPE == 'GPDO':
            raw   = GPDOParser.parse(_xml_path)
            entry = dict(timestamp=ts, fname=_fname, xml_path=_xml_path, raw=raw,
                         col=raw['col'], row=raw['row'],
                         lc_wl=raw['lc_wl'], fiber_dbm=raw['fiber_dbm'])
        else:
            data, root = MZMParser.parse_with_root(_xml_path)
            entry = dict(timestamp=ts, fname=_fname, xml_path=_xml_path,
                         raw=data, root=root,
                         col=data.get('Column'), row=data.get('Row'))
        parsed_data_list.append(entry)
        ok_count += 1
        _msg = f'({entry["col"]}, {entry["row"]})'
        if IS_ALL and _TQDM:
            _iter.set_postfix_str(_msg)
        else:
            print(f'  ✅ {_msg}')
    except Exception as _e:
        err_count += 1
        _msg = f'❌ {_fname}: {_e}'
        if IS_ALL and _TQDM:
            _tqdm.write(_msg)
        else:
            print(f'  {_msg}')

# ── Summary card ──────────────────────────────────────
_status_color  = '#aeffae' if err_count == 0 else '#ffd966'
_border_color  = '#2e5e2e' if err_count == 0 else '#5e4a00'
_bg_color      = '#1e2a1e' if err_count == 0 else '#2a2200'
_status_icon   = '✅' if err_count == 0 else '⚠'
_fail_line     = (f'<br><span style="color:#777;">Failed</span>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;'
                  f'<b style="color:#ffaaaa;">{err_count} file(s)</b>') if err_count else ''
_mode_label    = 'All Dies' if IS_ALL else 'Single Die'

display(_HTML(f'''
<div style="margin-top:12px; padding:12px 18px; border-radius:8px;
            background:{_bg_color}; border:1.5px solid {_border_color};
            font-size:13px; line-height:2.0;">
  <span style="font-size:14px; font-weight:bold; color:{_status_color};">
    {_status_icon} &nbsp;Parsing complete
  </span><br>
  <span style="color:#777;">Device</span>&nbsp;&nbsp;&nbsp;&nbsp;
    <b style="color:white;">{DEVICE_TYPE}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Wafer</span>&nbsp;&nbsp;
    <b style="color:white;">{WAFER_ID}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Timestamp</span>&nbsp;&nbsp;
    <b style="color:white;">{ts}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Mode</span>&nbsp;&nbsp;
    <b style="color:white;">{_mode_label}</b><br>
  <span style="color:#777;">Loaded</span>&nbsp;&nbsp;&nbsp;&nbsp;
    <b style="color:{_status_color};">{ok_count} / {len(xml_paths)} files</b>
  {_fail_line}
</div>'''))

---
## Step 4 — Analysis Results Table

<table style="width:60%; border-collapse:collapse; font-size:15px; margin-top:8px;">
  <thead>
    <tr style="background-color:#3a3a3a; color:white;">
      <th style="padding:10px 20px; text-align:center;">Mode</th>
      <th style="padding:10px 20px; text-align:center;">Visualization</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#555555; color:white;">
      <td style="padding:9px 20px; text-align:center;">Single Die</td>
      <td style="padding:9px 20px; text-align:center;">Result Table</td>
    </tr>
    <tr style="background-color:#666666; color:white;">
      <td style="padding:9px 20px; text-align:center;">All Dies</td>
      <td style="padding:9px 20px; text-align:center;">Total Result Table</td>
    </tr>
  </tbody>
</table>

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Step 4 : Fitting + Results Table
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from IPython.display import HTML as _HTML

result_rows   = []   # ★ Reset on every run
_row_statuses = []   # 'ok' | 'warn' | 'bad'  — used for row colour coding

# ── Quality thresholds ────────────────────────────────
# GPDO : R (A/W)  — Good ≥ 0.8 / Warn 0.5~0.8 / Bad < 0.5
#        n        — Good 1.0~1.5 / Warn 1.5~2.0 / Bad > 2.0 or < 1.0
# MZM  : ER (dB)  — Good ≥ 30   / Warn 20~30   / Bad < 20
#        n        — Good 1.0~1.8 / Warn 1.8~2.0 / Bad > 2.0 or < 1.0
_STATUS_RANK = {'ok': 0, 'warn': 1, 'bad': 2}

def _gpdo_status(R, n):
    if R is None or n is None or __import__('math').isnan(float(R)) or __import__('math').isnan(float(n)):
        return 'bad'
    R, n = float(R), float(n)
    s_R = 'ok' if R >= 0.8        else ('warn' if R >= 0.5 else 'bad')
    s_n = 'ok' if 1.0 <= n <= 1.5 else ('warn' if n <= 2.0 else 'bad')
    return max(s_R, s_n, key=lambda s: _STATUS_RANK[s])

def _mzm_status(ER, n):
    if ER is None or n is None or __import__('math').isnan(float(ER)) or __import__('math').isnan(float(n)):
        return 'bad'
    ER, n = float(ER), float(n)
    s_ER = 'ok' if ER >= 30        else ('warn' if ER >= 20  else 'bad')
    s_n  = 'ok' if 1.0 <= n <= 1.8 else ('warn' if n  <= 2.0 else 'bad')
    return max(s_ER, s_n, key=lambda s: _STATUS_RANK[s])

# ── Fitting & row building ────────────────────────────
for entry in parsed_data_list:
    raw = entry['raw']
    col = entry['col']; row = entry['row']; ts = entry['timestamp']

    if DEVICE_TYPE == 'GPDO':
        try:
            ref_r = FittingEngine.fit_reference(raw['L_ref'], raw['IL_ref'])
            df    = FittingEngine.fit_dark_fwd(raw['V_dark'], raw['I_dark'])
            dr    = FittingEngine.fit_dark_rev(raw['V_dark'], raw['I_dark'])
            lf    = FittingEngine.fit_light(raw['V_light'], raw['I_light'], df, dr)
            pc    = FittingEngine.calc_photo_current(
                        raw['V_light'], raw['I_light'], raw['V_dark'], raw['I_dark'])
            idx_wl  = np.argmin(np.abs(raw['L_ref'] - raw['lc_wl']))
            resp    = FittingEngine.calc_responsivity(pc['Iph'], raw['fiber_dbm'], raw['IL_ref'][idx_wl])
            peak_wl = float(raw['L_spec'][np.argmax(np.abs(raw['I_spec']))]) if len(raw['L_spec']) > 0 else float('nan')
            entry['fit_results'] = dict(ref_r=ref_r, df=df, dr=dr, lf=lf, pc=pc, resp=resp, peak_wl=peak_wl)
            result_rows.append({
                'Die (Col,Row)': f'({col},{row})',
                'λc (nm)'      : f"{raw['lc_wl']:.1f}",
                'Fiber (dBm)'  : f"{raw['fiber_dbm']:.2f}",
                'Iph (A)'      : f"{pc['Iph']:.3e}",
                'n'            : f"{df['n']:.4f}",
                'R (A/W)'      : f"{resp['R_resp']:.4f}",
                'Peak λ (nm)'  : f"{peak_wl:.1f}" if not __import__('math').isnan(peak_wl) else 'N/A',
            })
            _row_statuses.append(_gpdo_status(resp['R_resp'], df['n']))
        except Exception as e:
            print(f'Fitting failed [{entry["fname"]}]: {e}')
    else:
        er_val = raw.get('Extinction Ratio (dB)')
        n_val  = raw.get('Ideality Factor')
        result_rows.append({
            'Die (Col,Row)' : f'({col},{row})',
            'TestSite'      : raw.get('Testsite', 'N/A'),
            'Analysis λ'    : raw.get('Analysis Wavelength', 'N/A'),
            'I@-1V (A)'     : f"{raw['I at -1V [A]']:.3e}" if raw.get('I at -1V [A]') is not None else 'N/A',
            'Ideality n'    : f"{n_val:.4f}"                if n_val  is not None else 'N/A',
            'FSR (nm)'      : f"{raw['FSR (nm)']:.4f}"      if raw.get('FSR (nm)') is not None else 'N/A',
            'ER (dB)'       : f"{er_val:.2f}"               if er_val is not None else 'N/A',
        })
        _row_statuses.append(_mzm_status(er_val, n_val))

# ── Colour maps ───────────────────────────────────────
_ROW_BG = {
    ('ok',   0): '#4a4a4a', ('ok',   1): '#555555',
    ('warn', 0): '#5c4a00', ('warn', 1): '#6b5500',
    ('bad',  0): '#5c1414', ('bad',  1): '#6b1818',
}
_CELL_CSS = {
    'ok'  : 'background-color:#1a5c1a; color:#aeffae; font-weight:bold;',
    'warn': 'background-color:#7a5a00; color:#ffd966; font-weight:bold;',
    'bad' : 'background-color:#7a1010; color:#ffaaaa; font-weight:bold;',
}

def _cell_status_gpdo(col_name, val_str):
    try:
        v = float(val_str)
    except (ValueError, TypeError):
        return 'bad'
    if col_name == 'R (A/W)':
        return 'ok' if v >= 0.8 else ('warn' if v >= 0.5 else 'bad')
    if col_name == 'n':
        return 'ok' if 1.0 <= v <= 1.5 else ('warn' if v <= 2.0 else 'bad')
    return None

def _cell_status_mzm(col_name, val_str):
    try:
        v = float(val_str)
    except (ValueError, TypeError):
        return 'bad'
    if col_name == 'ER (dB)':
        return 'ok' if v >= 30 else ('warn' if v >= 20 else 'bad')
    if col_name == 'Ideality n':
        return 'ok' if 1.0 <= v <= 1.8 else ('warn' if v <= 2.0 else 'bad')
    return None

if DEVICE_TYPE == 'GPDO':
    _KEY_COLS    = {'n', 'R (A/W)'}
    _cell_status = _cell_status_gpdo
else:
    _KEY_COLS    = {'Ideality n', 'ER (dB)'}
    _cell_status = _cell_status_mzm

def _apply_styles(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for idx in df.index:
        st  = _row_statuses[idx] if idx < len(_row_statuses) else 'ok'
        bg  = _ROW_BG.get((st, idx % 2), '#555555')
        row_css = f'background-color:{bg}; color:white;'
        for col in df.columns:
            if col in _KEY_COLS:
                cell_st = _cell_status(col, df.at[idx, col])
                styles.at[idx, col] = _CELL_CSS.get(cell_st, row_css)
            else:
                styles.at[idx, col] = row_css
    return styles

# ── Display table ─────────────────────────────────────
_df = pd.DataFrame(result_rows)
display(
    _df.style
    .apply(_apply_styles, axis=None)
    .set_table_styles([
        {'selector': 'table',
         'props': [('width', '100%'), ('border-collapse', 'collapse')]},
        {'selector': 'th',
         'props': [('font-size', '14px'), ('font-weight', 'bold'),
                   ('background-color', '#3a3a3a'), ('color', 'white'),
                   ('padding', '8px 14px'), ('text-align', 'center'),
                   ('white-space', 'nowrap')]},
        {'selector': 'td',
         'props': [('font-size', '13px'), ('padding', '6px 14px'),
                   ('text-align', 'center'), ('white-space', 'nowrap')]},
    ])
    .hide(axis='index')
)

# ── Quality legend ────────────────────────────────────
if DEVICE_TYPE == 'GPDO':
    _metric_desc = 'R (A/W) + Ideality n'
    _leg_ok   = '✅ Good — R ≥ 0.8  &amp;  1.0 ≤ n ≤ 1.5'
    _leg_warn = '🟡 Warn — R 0.5~0.8  or  n 1.5~2.0'
    _leg_bad  = '🔴 Poor — R &lt; 0.5  or  n &gt; 2.0'
else:
    _metric_desc = 'ER (dB) + Ideality n'
    _leg_ok   = '✅ Good — ER ≥ 30 dB  &amp;  1.0 ≤ n ≤ 1.8'
    _leg_warn = '🟡 Warn — ER 20~30 dB  or  n 1.8~2.0'
    _leg_bad  = '🔴 Poor — ER &lt; 20 dB  or  n &gt; 2.0'

display(_HTML(f'''
<div style="margin-top:10px; font-size:12px; display:flex; gap:8px; align-items:center; flex-wrap:wrap;">
  <span style="color:#aaa; margin-right:4px;">Quality indicator ({_metric_desc}) :</span>
  <span style="background:#1a5c1a; color:#aeffae; padding:3px 12px; border-radius:4px;">{_leg_ok}</span>
  <span style="background:#7a5a00; color:#ffd966; padding:3px 12px; border-radius:4px;">{_leg_warn}</span>
  <span style="background:#7a1010; color:#ffaaaa; padding:3px 12px; border-radius:4px;">{_leg_bad}</span>
</div>
'''))

---
## Step 5 — Visualization

<table style="width:60%; border-collapse:collapse; font-size:15px; margin-top:8px;">
  <thead>
    <tr style="background-color:#3a3a3a; color:white;">
      <th style="padding:10px 20px; text-align:center;">Mode</th>
      <th style="padding:10px 20px; text-align:center;">Visualization</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#555555; color:white;">
      <td style="padding:9px 20px; text-align:center;">Single Die</td>
      <td style="padding:9px 20px; text-align:center;">6-panel Graph</td>
    </tr>
    <tr style="background-color:#666666; color:white;">
      <td style="padding:9px 20px; text-align:center;">All Dies</td>
      <td style="padding:9px 20px; text-align:center;">Heatmap + Raincloud Boxplot</td>
    </tr>
  </tbody>
</table>

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Step 5 : Graph Visualization
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from unittest.mock import patch
from IPython.display import display as ipy_display
from src.boxplot import _raincloud_ax
from src.gpdo.analyzer import GPDOAnalyzer as _GPDOAnalyzer

# ── Dependency check ──────────────────────────────────
try:
    _ = parsed_data_list, DEVICE_TYPE, WAFER_ID
except NameError as _e:
    raise RuntimeError(f'Variable not found ({_e}) — run Step 3 first.')
if not parsed_data_list:
    raise RuntimeError('No data loaded — run Step 3 first.')

IS_ALL = (selection.get('die') == 'All')

# ── Auto-fit for GPDO if Step 4 was skipped ──────────
if DEVICE_TYPE == 'GPDO':
    _missing = [e for e in parsed_data_list if e.get('fit_results') is None]
    if _missing:
        print(f'⚠ fit_results missing for {len(_missing)} entries — auto-fitting...')
        for _entry in _missing:
            _raw = _entry['raw']
            try:
                _ref_r = FittingEngine.fit_reference(_raw['L_ref'], _raw['IL_ref'])
                _df2   = FittingEngine.fit_dark_fwd(_raw['V_dark'], _raw['I_dark'])
                _dr    = FittingEngine.fit_dark_rev(_raw['V_dark'], _raw['I_dark'])
                _lf    = FittingEngine.fit_light(_raw['V_light'], _raw['I_light'], _df2, _dr)
                _pc    = FittingEngine.calc_photo_current(
                             _raw['V_light'], _raw['I_light'], _raw['V_dark'], _raw['I_dark'])
                _idx   = np.argmin(np.abs(_raw['L_ref'] - _raw['lc_wl']))
                _resp  = FittingEngine.calc_responsivity(_pc['Iph'], _raw['fiber_dbm'], _raw['IL_ref'][_idx])
                _pw    = float(_raw['L_spec'][np.argmax(np.abs(_raw['I_spec']))]) if len(_raw['L_spec']) > 0 else float('nan')
                _entry['fit_results'] = dict(ref_r=_ref_r, df=_df2, dr=_dr, lf=_lf,
                                             pc=_pc, resp=_resp, peak_wl=_pw)
            except Exception as _e2:
                print(f'  ❌ Auto-fit failed [{_entry["fname"]}]: {_e2}')
        print(f'  ✅ Auto-fit complete ({len(_missing)} entries)\n')

# ── Heatmap helper ────────────────────────────────────
def _heatmap_on_ax(ax, cols, rows, vals, title, unit, cmap, wafer_id):
    valid   = [(int(c), int(r), float(v))
               for c, r, v in zip(cols, rows, vals)
               if v is not None and not np.isnan(float(v))]
    missing = [(int(c), int(r))
               for c, r, v in zip(cols, rows, vals)
               if v is None or np.isnan(float(v))]
    if not valid:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                ha='center', va='center', color='gray', fontsize=13)
        ax.set_title(title, fontweight='bold'); return
    cs, rs, vs = zip(*valid)
    norm = Normalize(vmin=min(vs), vmax=max(vs))
    sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    for c, r, v in zip(cs, rs, vs):
        rect = mpatches.FancyBboxPatch(
            (c-0.45, r-0.45), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            facecolor=sm.to_rgba(v), edgecolor='white', linewidth=2)
        ax.add_patch(rect)
        txt_color = 'white' if norm(v) < 0.5 else 'black'
        ax.text(c, r, f'{v:.3g}', ha='center', va='center',
                fontsize=11, fontweight='bold', color=txt_color)
    for c, r in missing:
        rect = mpatches.FancyBboxPatch(
            (c-0.45, r-0.45), 0.9, 0.9,
            boxstyle='round,pad=0.05',
            facecolor='#EEEEEE', edgecolor='gray',
            linewidth=1, linestyle='--')
        ax.add_patch(rect)
        ax.text(c, r, 'N/A', ha='center', va='center', fontsize=9, color='gray')
    all_c = list(cs) + [c for c,_ in missing]
    all_r = list(rs) + [r for _,r in missing]
    ax.set_xlim(min(all_c)-0.9, max(all_c)+0.9)
    ax.set_ylim(min(all_r)-0.9, max(all_r)+0.9)
    ax.set_aspect('equal')
    unit_str = f' [{unit}]' if unit else ''
    ax.set_xlabel('Column (Die X)', fontsize=11)
    ax.set_ylabel('Row (Die Y)', fontsize=11)
    ax.set_title(f'Wafer {wafer_id} – {title}{unit_str}', fontweight='bold', fontsize=13)
    ax.tick_params(labelsize=10)
    ax.grid(True, alpha=0.15, linestyle=':')
    plt.colorbar(sm, ax=ax, label=f'{title}{unit_str}', shrink=0.85).ax.tick_params(labelsize=10)


def _run_and_show(plotter_fn):
    """Capture figures created by plotter (which calls plt.close internally) and display once."""
    intercepted = []
    def _intercept(fig=None):
        if fig is not None and hasattr(fig, 'axes'):
            intercepted.append(fig)
    before = set(plt.get_fignums())
    plt.ioff()
    try:
        with patch('matplotlib.pyplot.close', side_effect=_intercept):
            plotter_fn()
    finally:
        plt.ion()
    for fig in intercepted:
        ipy_display(fig); plt.close(fig)
    for n in sorted(set(plt.get_fignums()) - before):
        fig = plt.figure(n); ipy_display(fig); plt.close(fig)


# ════════════════════════════════════════════════
# ① Single Die
# ════════════════════════════════════════════════
if not IS_ALL:
    if DEVICE_TYPE == 'GPDO':
        from src.gpdo.plotter import Plotter as GPDOPlotter
        for entry in parsed_data_list:
            fr = entry.get('fit_results')
            if fr is None:
                print('fit_results not found — run Step 4 first'); continue
            _data_errors = _GPDOAnalyzer._check_data_quality(entry['raw'])
            if _data_errors:
                print(f'  ⚠ Data quality issues [{entry["fname"]}]: {_data_errors}')
            _run_and_show(lambda e=entry, f=fr, errs=_data_errors: GPDOPlotter.plot(
                e['raw'], f['ref_r'], f['df'], f['dr'], f['lf'], f['pc'], f['resp'],
                save_dir=None, wafer_id=WAFER_ID, fname=e['fname'],
                error_messages=errs if errs else None,
            ))
    else:
        from src.mzm.plotter import Plotter as MZMPlotter
        for entry in parsed_data_list:
            if entry.get('root') is None:
                print('XML root not found'); continue
            _run_and_show(lambda e=entry: MZMPlotter.plot_from_root(
                e['root'], e['fname'],
                save_dir=None, verbose=True, extra_info=e['raw'],
            ))

# ════════════════════════════════════════════════
# ② All — heatmap + Raincloud boxplot
# ════════════════════════════════════════════════
else:
    cols_all = [e['col'] for e in parsed_data_list]
    rows_all = [e['row'] for e in parsed_data_list]

    if DEVICE_TYPE == 'GPDO':
        R_vals   = [e['fit_results']['resp']['R_resp'] if e.get('fit_results') else None for e in parsed_data_list]
        n_vals   = [e['fit_results']['df']['n']        if e.get('fit_results') else None for e in parsed_data_list]
        Iph_vals = [e['fit_results']['pc']['Iph']      if e.get('fit_results') else None for e in parsed_data_list]

        fig, axes = plt.subplots(1, 2, figsize=(18, 8))
        fig.suptitle(f'GPDO Heatmap — {WAFER_ID} / {ts}', fontsize=16, fontweight='bold')
        _heatmap_on_ax(axes[0], cols_all, rows_all, R_vals,  'Responsivity',    'A/W', 'RdYlGn',  WAFER_ID)
        _heatmap_on_ax(axes[1], cols_all, rows_all, n_vals,  'Ideality Factor', '',    'coolwarm', WAFER_ID)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

        fig, axes = plt.subplots(1, 3, figsize=(22, 7))
        fig.suptitle(f'GPDO Raincloud — {WAFER_ID} / {ts}', fontsize=16, fontweight='bold')
        for ax, vals, ylabel in zip(axes,
            [R_vals, n_vals, Iph_vals],
            ['Responsivity [A/W]', 'Ideality Factor', 'Photo Current [A]']):
            arr = np.array([float(v) for v in vals if v is not None], dtype=float)
            _raincloud_ax(ax, {WAFER_ID: arr}, ylabel)
            ax.tick_params(labelsize=11); ax.yaxis.label.set_size(12)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

    else:  # MZM
        rows_data = [e['raw'] for e in parsed_data_list]
        hm_params = [
            ('Max transmission of Ref. spec (dB)', 'Max Transmission', 'dB',  'RdYlGn'),
            ('I at -1V [A]',                       'I at -1V',         'A',   'RdYlBu'),
            ('Extinction Ratio (dB)',               'Extinction Ratio', 'dB',  'RdYlGn'),
            ('FSR (nm)',                            'FSR',              'nm',  'RdYlBu'),
        ]
        fig, axes = plt.subplots(1, 4, figsize=(32, 8))
        fig.suptitle(f'MZM Heatmap — {WAFER_ID} / {ts}', fontsize=16, fontweight='bold')
        for ax, (key, title, unit, cmap) in zip(axes, hm_params):
            _heatmap_on_ax(ax, cols_all, rows_all, [r.get(key) for r in rows_data], title, unit, cmap, WAFER_ID)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

        bp_params = [
            ('Extinction Ratio (dB)', 'Extinction Ratio [dB]'),
            ('FSR (nm)',              'FSR [nm]'),
            ('I at -1V [A]',         'I at -1V [A]'),
            ('Ideality Factor',      'Ideality Factor'),
        ]
        fig, axes = plt.subplots(1, 4, figsize=(28, 7))
        fig.suptitle(f'MZM Boxplot — {WAFER_ID} / {ts}', fontsize=16, fontweight='bold')
        for ax, (key, ylabel) in zip(axes, bp_params):
            arr = np.array([float(r[key]) for r in rows_data if r.get(key) is not None], dtype=float)
            _raincloud_ax(ax, {WAFER_ID: arr}, ylabel)
            ax.tick_params(labelsize=11); ax.yaxis.label.set_size(12)
        plt.tight_layout(); ipy_display(fig); plt.close(fig)

print('Done')

---
## Step 6 — Save Results (CSV / PNG)

<table style="width:65%; border-collapse:collapse; font-size:15px; margin-top:8px;">
  <thead>
    <tr style="background-color:#3a3a3a; color:white;">
      <th style="padding:10px 20px; text-align:center;">Mode</th>
      <th style="padding:10px 20px; text-align:center;">Output</th>
    </tr>
  </thead>
  <tbody>
    <tr style="background-color:#555555; color:white;">
      <td style="padding:9px 20px; text-align:center;">Single Die</td>
      <td style="padding:9px 20px; text-align:center;">Result CSV + Graph PNG</td>
    </tr>
    <tr style="background-color:#666666; color:white;">
      <td style="padding:9px 20px; text-align:center;">All Dies</td>
      <td style="padding:9px 20px; text-align:center;">Total Result CSV + Figure PNG</td>
    </tr>
  </tbody>
</table>

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Step 6 : Save CSV / PNG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import math
import matplotlib.pyplot as _plt
import matplotlib.patches as _mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from IPython.display import FileLink, HTML as _HTML
from src.gpdo.plotter import Plotter as GPDOPlotter
from src.mzm.plotter  import Plotter as MZMPlotter
from src.mzm.csv      import save_rows_to_csv as mzm_save_rows
from src.boxplot      import _raincloud_ax

IS_ALL = (selection.get('die') == 'All')
ts     = selection['timestamp']

csv_dir     = os.path.join(RES_DIR, 'csv', DEVICE_TYPE, PROJECT_NAME)
png_dir     = os.path.join(RES_DIR, 'png', DEVICE_TYPE, PROJECT_NAME, WAFER_ID, ts)
heatmap_dir = os.path.join(png_dir, 'heatmap')
boxplot_dir = os.path.join(png_dir, 'boxplot')
os.makedirs(csv_dir, exist_ok=True)
os.makedirs(png_dir, exist_ok=True)

_saved_links = []  # (label, abs_path)
def _add_link(label, path): _saved_links.append((label, path))

# ════════════════════════════════════════════════
# Save CSV
# ════════════════════════════════════════════════
if DEVICE_TYPE == 'GPDO':
    csv_rows = []
    for entry in parsed_data_list:
        fr  = entry.get('fit_results')
        raw = entry['raw']
        if fr is None:
            print(f'  ⚠ No fit results [{entry["fname"]}] — Run Step 4 first')
            continue
        csv_rows.append({
            'wafer_id'  : WAFER_ID,
            'timestamp' : ts,
            'col'       : entry['col'],
            'row'       : entry['row'],
            'lc_wl'     : raw['lc_wl'],
            'peak_wl'   : fr['peak_wl'],
            'fiber_dbm' : raw['fiber_dbm'],
            'Iph'       : fr['pc']['Iph'],
            'n_d'       : fr['df']['n'],
            'R_resp'    : fr['resp']['R_resp'],
        })
    if csv_rows:
        csv_path = os.path.join(csv_dir, f'{WAFER_ID}_Result.csv')
        pd.DataFrame(csv_rows).to_csv(csv_path, index=False, encoding='utf-8-sig')
        _add_link('📄 CSV', csv_path)
    else:
        print('  ⚠ No CSV rows to save')
else:
    mzm_rows = [dict(e['raw'], timestamp=ts) for e in parsed_data_list]
    if mzm_rows:
        csv_path = mzm_save_rows(WAFER_ID, mzm_rows, verbose=True)
        if csv_path:
            _add_link('📄 CSV', csv_path)

# ════════════════════════════════════════════════
# Save PNG — Single Die
# ════════════════════════════════════════════════
if not IS_ALL:
    if DEVICE_TYPE == 'GPDO':
        for entry in parsed_data_list:
            fr = entry.get('fit_results')
            if fr is None:
                print(f'  ⚠ No fit results [{entry["fname"]}]'); continue
            GPDOPlotter.plot(
                entry['raw'], fr['ref_r'], fr['df'], fr['dr'], fr['lf'], fr['pc'], fr['resp'],
                save_dir=png_dir, wafer_id=WAFER_ID, fname=entry['fname'],
            )
            _add_link(f'🖼 PNG', os.path.join(png_dir, entry['fname'].replace('.xml', '.png')))
    else:
        for entry in parsed_data_list:
            if entry.get('root') is None:
                print(f'  ⚠ XML root not found [{entry["fname"]}]'); continue
            out = MZMPlotter.plot_from_root(
                entry['root'], entry['fname'],
                save_dir=png_dir, verbose=True, extra_info=entry['raw'],
            )
            if out:
                _add_link('🖼 PNG', out)

# ════════════════════════════════════════════════
# Save PNG — All mode (heatmap + boxplot)
# ════════════════════════════════════════════════
else:
    os.makedirs(heatmap_dir, exist_ok=True)
    os.makedirs(boxplot_dir, exist_ok=True)
    cols_all = [e['col'] for e in parsed_data_list]
    rows_all = [e['row'] for e in parsed_data_list]

    def _heatmap_ax_save(ax, cols, rows, vals, title, unit, cmap, wafer_id):
        valid   = [(int(c), int(r), float(v)) for c, r, v in zip(cols, rows, vals)
                   if v is not None and not math.isnan(float(v))]
        missing = [(int(c), int(r)) for c, r, v in zip(cols, rows, vals)
                   if v is None or math.isnan(float(v))]
        if not valid:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes,
                    ha='center', va='center', color='gray')
            ax.set_title(title, fontweight='bold'); return
        cs, rs, vs = zip(*valid)
        norm = Normalize(vmin=min(vs), vmax=max(vs))
        sm   = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
        for c, r, v in zip(cs, rs, vs):
            ax.add_patch(_mpatches.FancyBboxPatch(
                (c-0.45, r-0.45), 0.9, 0.9, boxstyle='round,pad=0.05',
                facecolor=sm.to_rgba(v), edgecolor='white', linewidth=1.5))
            ax.text(c, r, f'{v:.3g}', ha='center', va='center', fontsize=8,
                    fontweight='bold', color='white' if norm(v) < 0.5 else 'black')
        for c, r in missing:
            ax.add_patch(_mpatches.FancyBboxPatch(
                (c-0.45, r-0.45), 0.9, 0.9, boxstyle='round,pad=0.05',
                facecolor='#EEEEEE', edgecolor='gray', linewidth=1, linestyle='--'))
            ax.text(c, r, 'N/A', ha='center', va='center', fontsize=7, color='gray')
        all_c = list(cs) + [c for c, _ in missing]
        all_r = list(rs) + [r for _, r in missing]
        ax.set_xlim(min(all_c)-0.7, max(all_c)+0.7)
        ax.set_ylim(min(all_r)-0.7, max(all_r)+0.7)
        ax.set_aspect('equal')
        unit_str = f' [{unit}]' if unit else ''
        ax.set_xlabel('Column (Die X)', fontsize=9)
        ax.set_ylabel('Row (Die Y)', fontsize=9)
        ax.set_title(f'Wafer {wafer_id} – {title}{unit_str}', fontweight='bold', fontsize=10)
        ax.grid(True, alpha=0.15, linestyle=':')
        _plt.colorbar(sm, ax=ax, label=f'{title}{unit_str}', shrink=0.85)

    if DEVICE_TYPE == 'GPDO':
        R_vals   = [e['fit_results']['resp']['R_resp'] if e.get('fit_results') else None for e in parsed_data_list]
        n_vals   = [e['fit_results']['df']['n']        if e.get('fit_results') else None for e in parsed_data_list]
        Iph_vals = [e['fit_results']['pc']['Iph']      if e.get('fit_results') else None for e in parsed_data_list]

        fig, axes = _plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'GPDO Heatmap — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        _heatmap_ax_save(axes[0], cols_all, rows_all, R_vals,  'Responsivity',    'A/W', 'RdYlGn',  WAFER_ID)
        _heatmap_ax_save(axes[1], cols_all, rows_all, n_vals,  'Ideality Factor', '',    'coolwarm', WAFER_ID)
        _plt.tight_layout()
        hm_path = os.path.join(heatmap_dir, f'{WAFER_ID}_heatmap.png')
        fig.savefig(hm_path, dpi=150, bbox_inches='tight'); _plt.close(fig)
        _add_link('🖼 Heatmap', hm_path)

        fig, axes = _plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'GPDO Boxplot — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        for ax, vals, ylabel in zip(axes, [R_vals, n_vals, Iph_vals],
                                    ['Responsivity [A/W]', 'Ideality Factor', 'Photo Current [A]']):
            arr = np.array([float(v) for v in vals if v is not None], dtype=float)
            _raincloud_ax(ax, {WAFER_ID: arr}, ylabel)
        _plt.tight_layout()
        bp_path = os.path.join(boxplot_dir, f'{WAFER_ID}_boxplot.png')
        fig.savefig(bp_path, dpi=150, bbox_inches='tight'); _plt.close(fig)
        _add_link('🖼 Boxplot', bp_path)

    else:
        rows_data = [e['raw'] for e in parsed_data_list]
        hm_params = [
            ('Max transmission of Ref. spec (dB)', 'Max Transmission', 'dB', 'RdYlGn'),
            ('I at -1V [A]',                       'I at -1V',         'A',  'RdYlBu'),
            ('Extinction Ratio (dB)',               'Extinction Ratio', 'dB', 'RdYlGn'),
            ('FSR (nm)',                            'FSR',              'nm', 'RdYlBu'),
        ]
        fig, axes = _plt.subplots(1, 4, figsize=(26, 6))
        fig.suptitle(f'MZM Heatmap — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        for ax, (key, title, unit, cmap) in zip(axes, hm_params):
            _heatmap_ax_save(ax, cols_all, rows_all, [r.get(key) for r in rows_data], title, unit, cmap, WAFER_ID)
        _plt.tight_layout()
        hm_path = os.path.join(heatmap_dir, f'{WAFER_ID}_heatmap.png')
        fig.savefig(hm_path, dpi=150, bbox_inches='tight'); _plt.close(fig)
        _add_link('🖼 Heatmap', hm_path)

        bp_params = [
            ('Extinction Ratio (dB)', 'Extinction Ratio [dB]'),
            ('FSR (nm)',              'FSR [nm]'),
            ('I at -1V [A]',         'I at -1V [A]'),
            ('Ideality Factor',      'Ideality Factor'),
        ]
        fig, axes = _plt.subplots(1, 4, figsize=(22, 5))
        fig.suptitle(f'MZM Boxplot — {WAFER_ID} / {ts}', fontsize=13, fontweight='bold')
        for ax, (key, ylabel) in zip(axes, bp_params):
            arr = np.array([float(r[key]) for r in rows_data if r.get(key) is not None], dtype=float)
            _raincloud_ax(ax, {WAFER_ID: arr}, ylabel)
        _plt.tight_layout()
        bp_path = os.path.join(boxplot_dir, f'{WAFER_ID}_boxplot.png')
        fig.savefig(bp_path, dpi=150, bbox_inches='tight'); _plt.close(fig)
        _add_link('🖼 Boxplot', bp_path)

# ── Summary card ──────────────────────────────────────
_file_rows = ''
for _label, _path in _saved_links:
    try:
        _rel = os.path.relpath(_path)
        _file_rows += (f'<a href="{_rel}" target="_blank" '
                       f'style="display:block; color:#aaccff; text-decoration:none; '
                       f'padding:2px 0; font-size:12px;">'
                       f'{_label} &nbsp;→&nbsp; {_rel}</a>')
    except Exception:
        _file_rows += f'<span style="color:#aaccff; font-size:12px;">{_label} → {_path}</span><br>'

display(_HTML(f'''
<div style="margin-top:12px; padding:12px 18px; border-radius:8px;
            background:#1a2233; border:1.5px solid #2a4a7a;
            font-size:13px; line-height:2.0;">
  <span style="font-size:14px; font-weight:bold; color:#aaccff;">
    ✅ &nbsp;Save complete
  </span><br>
  <span style="color:#777;">Device</span>&nbsp;&nbsp;&nbsp;&nbsp;
    <b style="color:white;">{DEVICE_TYPE}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Wafer</span>&nbsp;&nbsp;
    <b style="color:white;">{WAFER_ID}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Timestamp</span>&nbsp;&nbsp;
    <b style="color:white;">{ts}</b>
  &nbsp;│&nbsp;
  <span style="color:#777;">Files saved</span>&nbsp;&nbsp;
    <b style="color:#aaccff;">{len(_saved_links)}</b><br>
  <div style="margin-top:4px; padding-top:6px; border-top:1px solid #2a4a7a;">
    {_file_rows}
  </div>
</div>'''))